# Phase 5: Build a RAG system from scratch (Ocean Research Assistant)

**How to use this notebook:** run the cells **top to bottom, one at a time**. After each step there is a **✅ Checkpoint** cell: look at its output and make sure it makes sense before moving on.

```
PDFs → 1 load → 2 chunk → 3 label → 4 embed+store → 5 retrieve → 6 prompt → 7 cite → 8 "I don't know"
                                                                                          ↓
                                                              9 measure → 10 experiment
```

Put your PDFs in a folder called `data/` next to this notebook (or use the practice PDFs from Step 0).
Concept notes: `[[Phase 5 - Build RAG From Scratch]]` in your Obsidian vault.

## Phase 0: Setup
Works in **Jupyter, VS Code, or Google Colab** (Colab: upload your PDFs to the `data` folder in the left sidebar).
Run the install cell once. It can take a few minutes (PyTorch is big).

In [ ]:
%pip install -q pypdf sentence-transformers faiss-cpu numpy anthropic python-dotenv reportlab

### 0.1 Settings (the knobs)
Change **one knob at a time**, then re-run the scoring in Step 9. `MIN_SCORE` is the "too unrelated" cut-off used in Step 8.

Your Anthropic key: either put `ANTHROPIC_API_KEY=...` in a `.env` file next to this notebook, or just run the cell and paste it when asked (it is not saved in the notebook). **Never commit your key to Git.**

In [ ]:
import os, re, json, math
from pathlib import Path
from getpass import getpass

import numpy as np
import faiss
from pypdf import PdfReader
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")

# ---- the knobs ----
DATA_DIR      = Path("data")       # your PDFs go here
CHUNK_SIZE    = 800                # max characters per chunk
CHUNK_OVERLAP = 100                # characters shared by neighbouring chunks
EMBED_MODEL   = "all-MiniLM-L6-v2" # small, free, runs on your computer (384 numbers per chunk)
TOP_K         = 4                  # chunks given to the LLM
MIN_SCORE     = 0.30               # best chunk below this => say "I don't know" (no LLM call)
LLM_MODEL     = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")
MAX_ANSWER_TOKENS = 500

DATA_DIR.mkdir(exist_ok=True)
print("Ready. PDFs in data/:", [p.name for p in DATA_DIR.glob("*.pdf")])

## Step 0: Practice PDFs (optional)
No PDFs yet? This makes the 4-sentence "mini library" from Phase 4 so you can test the whole pipeline.
**Skip this cell once you have real PDFs** (or delete the practice ones, otherwise they get mixed into your results).

In [ ]:
MAKE_SAMPLES = False   # set True to create the 4 practice PDFs

if MAKE_SAMPLES:
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas
    DOCS = {
        "primer.pdf":    "Sea level anomaly is how much sea level differs from its long-term average.",
        "warnings.pdf":  "INCOIS issues tsunami warnings using seismic data and tide gauges.",
        "cyclones.pdf":  "Cyclones over the Bay of Bengal are tracked using satellite images.",
        "altimetry.pdf": "Satellite altimeters measure sea surface height from space.",
    }
    for name, text in DOCS.items():
        c = canvas.Canvas(str(DATA_DIR / name), pagesize=A4)
        c.setFont("Helvetica", 12); c.drawString(72, 750, text); c.save()
    print("Wrote practice PDFs:", list(DOCS))

## Step 1: Load text and page numbers
**Goal:** turn every PDF page into `{"text", "source", "page"}`. Pages with almost no text (scanned images) are skipped with a warning.

In [ ]:
def clean(text: str) -> str:
    """Tidy extracted text: join hyphenated line breaks, collapse extra spaces."""
    text = text.replace("-\n", "").replace("\r", "")
    lines = [" ".join(ln.split()) for ln in text.split("\n")]
    out = "\n".join(lines)
    while "\n\n\n" in out:
        out = out.replace("\n\n\n", "\n\n")
    return out.strip()

def load_pdf(path: Path) -> list[dict]:
    pages = []
    for i, page in enumerate(PdfReader(str(path)).pages, start=1):   # humans count from 1
        text = clean(page.extract_text() or "")
        if len(text) < 20:
            print(f"  [skip] {path.name} page {i}: almost no text (scanned image?)")
            continue
        pages.append({"text": text, "source": path.name, "page": i})
    return pages

def load_folder(folder: Path) -> list[dict]:
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        raise SystemExit(f"No PDFs found in {folder.resolve()}. Add some (or set MAKE_SAMPLES=True above).")
    pages = []
    for pdf in pdfs:
        got = load_pdf(pdf)
        print(f"Loaded {pdf.name}: {len(got)} pages with text")
        pages.extend(got)
    return pages

pages = load_folder(DATA_DIR)

**✅ Checkpoint 1:** you should see real sentences, not junk characters or words glued together.

In [ ]:
print(len(pages), "pages")
print(pages[0]["source"], "page", pages[0]["page"])
print(pages[0]["text"][:400])

## Steps 2-3: Chunk and label
**Goal:** pages → chunks of about `CHUNK_SIZE` characters with `CHUNK_OVERLAP` shared between neighbours. Each chunk keeps its `source` and `page` (the **label**), which later powers citations.

The splitter is *recursive*: try the biggest natural break first (paragraph), then line, then sentence, then word. Only cut harder if a piece is still too big.

In [ ]:
SEPARATORS = ["\n\n", "\n", ". ", " "]

def _split(text, size, seps):
    if len(text) <= size:
        return [text]
    if not seps:                                      # nothing left to try: hard cut
        return [text[i:i + size] for i in range(0, len(text), size)]
    sep, rest = seps[0], seps[1:]
    if sep not in text:
        return _split(text, size, rest)
    pieces, current = [], ""
    for part in text.split(sep):
        part = part + (sep if sep.strip() else "")     # keep the full stop, drop pure whitespace
        if len(part) > size:                           # this part alone is too big: go finer
            if current:
                pieces.append(current); current = ""
            pieces.extend(_split(part, size, rest))
        elif len(current) + len(part) <= size:
            current += part                            # still fits: glue on
        else:
            pieces.append(current); current = part     # full: start a new piece
    if current:
        pieces.append(current)
    return [p.strip() for p in pieces if p.strip()]

def split_text(text, size, overlap):
    pieces = _split(text, size, SEPARATORS)
    if overlap <= 0 or len(pieces) < 2:
        return pieces
    out = [pieces[0]]
    for prev, cur in zip(pieces, pieces[1:]):
        tail = prev[-overlap:]
        tail = tail[tail.find(" ") + 1:] if " " in tail else tail   # start at a word boundary
        out.append((tail + " " + cur).strip())
    return out

def chunk_pages(pages, size, overlap):
    chunks = []
    for p in pages:
        for j, piece in enumerate(split_text(p["text"], size, overlap)):
            chunks.append({"id": f'{p["source"]}::p{p["page"]}::c{j}',
                           "text": piece, "source": p["source"], "page": p["page"]})
    return chunks

chunks = chunk_pages(pages, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"{len(pages)} pages -> {len(chunks)} chunks")

**✅ Checkpoint 2:** do chunks end near sentence boundaries? Are lengths near the size you set (not 5 characters, not 3,000)? Does every chunk have a source and page?

In [ ]:
lengths = [len(c["text"]) for c in chunks]
print("chunk length  min/avg/max:", min(lengths), round(sum(lengths)/len(lengths)), max(lengths))
for c in chunks[:3]:
    print("\n", c["id"], f'({len(c["text"])} chars)')
    print(c["text"][:250])

## Step 4: Embed and store
**Goal:** one vector (384 numbers) per chunk. Vectors are *normalised* to length 1, so the inner product **is** cosine similarity (Phase 4). FAISS `IndexFlatIP` searches them exactly.

Row *i* of the index = `chunks[i]`. That is why we keep the `chunks` list next to the index.
The first run downloads the embedding model (~90 MB) once.

In [ ]:
from sentence_transformers import SentenceTransformer

class Embedder:
    def __init__(self, name=EMBED_MODEL):
        self.model = SentenceTransformer(name)
    def encode(self, texts):
        v = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=len(texts) > 50)
        return np.asarray(v, dtype="float32")

embedder = Embedder()

def build_index(chunks, embedder):
    vecs = embedder.encode([c["text"] for c in chunks])
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    return index

index = build_index(chunks, embedder)
print(f"Stored {index.ntotal} vectors of {index.d} numbers each")

**✅ Checkpoint 3:** `ntotal` must equal the number of chunks. Optionally **save** the index so you don't recompute next session (re-run ingest whenever PDFs or chunk settings change):

In [ ]:
assert index.ntotal == len(chunks)

INDEX_DIR = Path("index"); INDEX_DIR.mkdir(exist_ok=True)
def save_index():
    faiss.write_index(index, str(INDEX_DIR / "faiss.index"))
    (INDEX_DIR / "chunks.json").write_text(json.dumps(chunks, indent=1), encoding="utf-8")
def load_saved_index():
    idx = faiss.read_index(str(INDEX_DIR / "faiss.index"))
    ch = json.loads((INDEX_DIR / "chunks.json").read_text(encoding="utf-8"))
    assert idx.ntotal == len(ch), "index and chunks.json out of sync - rebuild"
    return idx, ch

save_index()
print("Saved to", INDEX_DIR.resolve(), "(to reload later: index, chunks = load_saved_index())")

## Step 5: Retrieve
**Goal:** question → top-k most similar chunks, with scores. The question must be embedded with the **same model**.

> **Test search BEFORE connecting the LLM.** If the right chunk is not in the top results, the LLM cannot save you.

In [ ]:
def retrieve(question, k=TOP_K, index=None, chunks_=None):
    index = index if index is not None else globals()["index"]
    chunks_ = chunks_ if chunks_ is not None else globals()["chunks"]
    qvec = embedder.encode([question])
    scores, ids = index.search(qvec, k)
    return [{**chunks_[i], "score": float(s)} for s, i in zip(scores[0], ids[0]) if i != -1]

def peek(question, k=TOP_K):
    """Show what search finds. Free: no LLM call."""
    for c in retrieve(question, k):
        print(f'{c["score"]:.2f}  {c["source"]} p{c["page"]}  {c["text"][:100]!r}')

**✅ Checkpoint 4:** try 5 questions whose answer you know. Is the right chunk near the top? (Edit the question below.)

In [ ]:
peek("How do satellites measure sea level?")

## Steps 6-7: Build the prompt, call the LLM, cite sources
- Each chunk gets a number: `[1] (file.pdf, page 7) text…` so the model can cite it.
- The **system prompt** sets the rules: only the sources, cite `[n]`, otherwise say "I don't know".
- `temperature=0` so answers repeat while you test.
- After the answer, we read `[1]`, `[2]` back out and map them to file + page.

In [ ]:
import anthropic

SYSTEM_PROMPT = """You are a research assistant that answers ONLY from the numbered sources provided.

Rules:
1. Use only facts that appear in the sources. Do not use outside knowledge.
2. After each claim, cite the source number like [1] or [2].
3. If the sources do not contain the answer, reply exactly: "I don't know based on the provided documents."
4. Be concise: 2 to 5 sentences unless the user asks for more."""

NO_ANSWER = "I don't know based on the provided documents."

def build_context(chunks_):
    return "\n\n".join(f'[{i}] ({c["source"]}, page {c["page"]})\n{c["text"]}'
                        for i, c in enumerate(chunks_, start=1))

def build_user_message(question, chunks_):
    return f"Sources:\n{build_context(chunks_)}\n\nQuestion: {question}"

def generate(question, chunks_):
    client = anthropic.Anthropic()                     # reads ANTHROPIC_API_KEY
    reply = client.messages.create(
        model=LLM_MODEL, max_tokens=MAX_ANSWER_TOKENS, temperature=0,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": build_user_message(question, chunks_)}],
    )
    return reply.content[0].text.strip()

def cited_sources(answer, chunks_):
    nums = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
    return [chunks_[n - 1] for n in nums if 1 <= n <= len(chunks_)]

## Step 8: Say "I don't know"
Two safety nets:
1. **Before the LLM:** if even the best chunk scores under `MIN_SCORE`, return "I don't know" without calling the model (cheaper and safer).
2. **Inside the prompt:** rule 3 tells the model to refuse when the sources lack the answer.

`ask()` below is the **whole RAG system** in one function. Read it top to bottom.

In [ ]:
def ask(question, k=TOP_K, min_score=MIN_SCORE, show_prompt=False):
    found = retrieve(question, k)
    if not found or found[0]["score"] < min_score:                       # Step 8a
        return {"answer": NO_ANSWER, "sources": [], "chunks": found, "llm_called": False}
    if show_prompt:
        print(build_user_message(question, found))
    answer = generate(question, found)                                    # Steps 6
    return {"answer": answer, "sources": cited_sources(answer, found),    # Step 7
            "chunks": found, "llm_called": True}

def show(result):
    print(result["answer"])
    if result["sources"]:
        print("\nSources:")
        for i, c in enumerate(result["chunks"], start=1):
            if c in result["sources"]:
                print(f'  [{i}] {c["source"]}, page {c["page"]}')
    if not result["llm_called"]:
        print("\n(no LLM call: best score was below MIN_SCORE)")

**✅ Checkpoint 5 (ask):** the answer should be correct and cite a file + page. **Open that page and verify it.** If the page doesn't say it, the system is lying: fix that first.

In [ ]:
show(ask("How do satellites measure sea level?"))

**✅ Checkpoint 6 (don't know):** ask 5 questions that are NOT in your documents. All should refuse. Then 5 that ARE; none should refuse.
If good questions get refused, **lower** `MIN_SCORE`. If bad ones get answered, raise it (use `peek` to see the scores).

In [ ]:
for q in ["Who won the 2011 cricket World Cup?",
          "What is the capital of France?"]:
    print("Q:", q); show(ask(q)); print("-" * 40)

**Debugging ladder (use it every time an answer is wrong):**
1. `peek(question)`: was the right chunk retrieved? **No** → fix loading/chunking/search.
2. Yes → `ask(question, show_prompt=True)`: is the chunk inside the prompt?
3. Yes → the prompt or model is the problem. Adjust the rules and rerun.

## Step 9: Measure (the evaluation set)
Each entry says: *for this question, the answer is in this file and page.* A question is a **hit@k** if a chunk from that file+page is in the top *k*. **MRR** is higher when the right chunk sits nearer the top (1/rank of the first correct chunk, averaged).

**Replace the example entries with your real 20+ questions** (easy, medium, hard, and a few whose answer is *not* in any document). You can also save them as `eval_set.json` next to this notebook and they will be loaded instead.

In [ ]:
EVAL_SET = [
    {"question": "How do satellites measure sea level?", "source": "altimetry.pdf", "page": 1},
    {"question": "What is sea level anomaly?",           "source": "primer.pdf",    "page": 1},
    {"question": "How are tsunami warnings issued?",     "source": "warnings.pdf",  "page": 1},
    {"question": "How are cyclones tracked?",            "source": "cyclones.pdf",  "page": 1},
]
if Path("eval_set.json").exists():
    EVAL_SET = json.loads(Path("eval_set.json").read_text(encoding="utf-8"))
print(len(EVAL_SET), "evaluation questions")

In [ ]:
KS = [1, 3, 5]

def first_hit_rank(results, item):
    for rank, c in enumerate(results, start=1):
        if c["source"] == item["source"] and c["page"] == item["page"]:
            return rank
    return None

def evaluate(index_=None, chunks_=None, verbose=True):
    ranks = []
    for item in EVAL_SET:
        res = retrieve(item["question"], max(KS), index_, chunks_)
        rank = first_hit_rank(res, item)
        ranks.append(rank)
        if verbose and (rank is None or rank > 3):
            got = [f'{c["source"]} p{c["page"]}' for c in res[:3]]
            print(f'MISS  {item["question"]!r}\n      wanted {item["source"]} p{item["page"]}, got {got}')
    n = len(EVAL_SET)
    row = {f"hit@{k}": sum(1 for r in ranks if r and r <= k) / n for k in KS}
    row["MRR"] = sum(1 / r for r in ranks if r) / n
    return row

row = evaluate()
print(f"\nchunk_size={CHUNK_SIZE} overlap={CHUNK_OVERLAP} top_k={TOP_K}")
print({k: round(v, 2) for k, v in row.items()})

Reading it: **hit@3 = 0.80** means the right chunk was in the top 3 for 80% of questions. Each `MISS` is a to-do: `peek` it and ask *was the text split badly? is the question worded very differently from the text?*

## Step 10: Experiments (one change at a time)
This cell rebuilds the index in memory for each setting (no need to edit knobs and re-run everything) and prints a table. **Copy the rows into your project log** ( `[[Ocean Research Assistant - Project Log]]` ).
Change `SETTINGS` to whatever you want to try. Keep top-k fixed while you compare chunk sizes.

In [ ]:
SETTINGS = [  # (chunk_size, overlap)
    (800, 100),   # baseline
    (400, 50),    # smaller chunks
    (1200, 150),  # bigger chunks
    (800, 0),     # does overlap matter?
]

rows = []
for size, ov in SETTINGS:
    ch = chunk_pages(pages, size, ov)
    idx = build_index(ch, embedder)
    r = evaluate(idx, ch, verbose=False)
    rows.append((size, ov, len(ch), r))

print("| chunk/overlap | chunks | hit@1 | hit@3 | hit@5 | MRR |")
print("|---|---|---|---|---|---|")
for size, ov, n, r in rows:
    print(f'| {size}/{ov} | {n} | {r["hit@1"]:.2f} | {r["hit@3"]:.2f} | {r["hit@5"]:.2f} | {r["MRR"]:.2f} |')

**Then top-k:** keep the best chunk settings and compare `TOP_K` = 2, 4, 8 by reading **5 full answers** each (`ask(q, k=...)`). More chunks help recall but add noise and cost.

**Write one sentence you can say in an interview:** *"Going from 800 to 400-character chunks raised hit@3 from X to Y because …"*

In [ ]:
# Compare top-k by eye on one question
q = "How do satellites measure sea level?"
for k in [2, 4, 8]:
    print(f"--- k={k} ---"); show(ask(q, k=k))

## Common problems
| Symptom | Likely cause | Fix |
|---|---|---|
| `No PDFs found` | files not in `data/` | check folder and `.pdf` extension |
| "[skip] almost no text" | scanned PDF | OCR it first, or use another PDF |
| Right text exists but never retrieved | chunks too big/small, or the question is worded very differently | `peek`, try another size; Phase 7 adds keyword/hybrid search |
| `AuthenticationError` | key missing or wrong | re-run the settings cell and paste the key again |
| Everything refused | `MIN_SCORE` too high | `peek` the scores and lower it |
| Answer cites an irrelevant `[3]` | model sloppy | tighten the system prompt, lower `TOP_K` |
| Changed PDFs but results look old | index not rebuilt | re-run Steps 1-4 |
| Swapped the embedding model | old vectors are in a different space | re-embed everything |

## Done when
- [ ] `ask()` answers 5 hand-verified questions with the correct file and page
- [ ] Out-of-scope questions return "I don't know"
- [ ] `evaluate()` runs on a 20-question set and you logged a baseline row
- [ ] You ran at least 3 experiments and wrote what you learned
- [ ] You can explain each step without looking at the code

**Next:** Phase 6 rebuilds this in LangChain. Your scores there should match this baseline.